# Project 3 · Advanced Solution — Single Inheritance

**Learning goals:** implement `Resource → CPU` and `Resource → Storage → HDD/SSD`,
validate state, enforce read-only public properties, eliminate duplication with
`super()`, and test inventory invariants. **Runtime:** modern Python 3.11+;
**dependencies:** standard library only.

This notebook is **standalone**: run cells top-to-bottom in a fresh kernel.
The original project mentions `SDD` in two places; the correct class is `SSD`.

## Design contract and important choices

- A resource represents **one kind of component** and a pool of identical items.
- `0 <= allocated <= total`; `available = total - allocated`. Counts start at
  the **current** supplied state, rather than describing an initial purchase.
- `claim(n)` allocates available inventory; `freeup(n)` returns working allocated
  items; `died(n)` removes **damaged allocated** items from both counts;
  `purchased(n)` increases total. All methods require a strictly positive integer.
- A failed operation changes nothing. `bool` is rejected even though it inherits
  from `int`. Public configuration and counts are property read-only.
- An `RLock` prevents overlapping methods on the **same resource** from
  breaking its invariant. Consistent snapshots are immutable.
- `__repr__` names all constructor arguments and can reconstruct an instance
  in a controlled environment; do **not** use `eval` on untrusted strings.
- Python underscores/private storage are an API convention, not tamper-proof
  security. The expanded second notebook introduces a multi-SKU catalog.

## 1 · Shared validation and immutable snapshots

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from threading import RLock
from typing import Any


def validate_integer(
    value: object,
    *,
    name: str,
    minimum: int | None = None,
    maximum: int | None = None,
    bounds_message: str | None = None,
) -> int:
    """Return an integer after strict type and inclusive bound checks.

    Booleans are rejected: ``bool`` is an ``int`` subclass but not a count.
    """
    if type(value) is not int:
        raise TypeError(f"{name} must be an integer (bool is not accepted)")
    if minimum is not None and value < minimum:
        raise ValueError(bounds_message or f"{name} must be >= {minimum}")
    if maximum is not None and value > maximum:
        raise ValueError(bounds_message or f"{name} must be <= {maximum}")
    return value


def validate_text(value: object, *, name: str) -> str:
    """Accept a non-blank string, stripping surrounding whitespace."""
    if not isinstance(value, str):
        raise TypeError(f"{name} must be a string")
    result = value.strip()
    if not result:
        raise ValueError(f"{name} must not be blank")
    return result


@dataclass(frozen=True, slots=True)
class InventorySnapshot:
    """An immutable, consistent view of the stock at one instant."""

    name: str
    category: str
    manufacturer: str
    total: int
    allocated: int
    available: int

## 2 · Base class: Resource

In [2]:
class Resource:
    """A stock pool for one type of hardware, not an individual physical item.

    Invariant: 0 <= allocated <= total. Public configuration is read-only.
    ``died(n)`` retires n *allocated* damaged items; call ``freeup(n)``
    instead for working items returned to stock.
    """

    __slots__ = ("_name", "_manufacturer", "_total", "_allocated", "_lock")

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        name = validate_text(name, name="name")
        manufacturer = validate_text(manufacturer, name="manufacturer")
        total = validate_integer(total, name="total", minimum=0)
        allocated = validate_integer(allocated, name="allocated", minimum=0)
        if allocated > total:
            raise ValueError("allocated cannot exceed total")
        self._name = name
        self._manufacturer = manufacturer
        self._total = total
        self._allocated = allocated
        self._lock = RLock()

    @property
    def name(self) -> str:
        return self._name

    @property
    def manufacturer(self) -> str:
        return self._manufacturer

    @property
    def manufacturer_name(self) -> str:
        """Compatibility alias for exercise wording."""
        return self._manufacturer

    @property
    def total(self) -> int:
        with self._lock:
            return self._total

    @property
    def allocated(self) -> int:
        with self._lock:
            return self._allocated

    @property
    def available(self) -> int:
        with self._lock:
            return self._total - self._allocated

    @property
    def category(self) -> str:
        return type(self).__name__.lower()

    def snapshot(self) -> InventorySnapshot:
        with self._lock:
            return InventorySnapshot(
                name=self.name,
                category=self.category,
                manufacturer=self.manufacturer,
                total=self._total,
                allocated=self._allocated,
                available=self._total - self._allocated,
            )

    def _check_quantity(self, quantity: int) -> int:
        return validate_integer(quantity, name="quantity", minimum=1)

    def claim(self, n: int) -> None:
        """Allocate n available items; reject the entire request if short."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._total - self._allocated:
                raise ValueError("insufficient available inventory")
            self._allocated += n

    def freeup(self, n: int) -> None:
        """Return n allocated, working items to available stock."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._allocated:
                raise ValueError("cannot free more items than are allocated")
            self._allocated -= n

    def died(self, n: int) -> None:
        """Retire n damaged allocated items (total and allocated decrease)."""
        n = self._check_quantity(n)
        with self._lock:
            if n > self._allocated:
                raise ValueError("cannot retire more damaged items than allocated")
            self._allocated -= n
            self._total -= n

    def purchased(self, n: int) -> None:
        """Add n newly purchased items to available inventory."""
        n = self._check_quantity(n)
        with self._lock:
            self._total += n

    def _spec_repr(self) -> str:
        """Extension point for subclass-specific constructor arguments."""
        return ""

    def __repr__(self) -> str:
        with self._lock:
            extras = self._spec_repr()
            if extras:
                extras = ", " + extras
            return (
                f"{type(self).__name__}(name={self.name!r}, "
                f"manufacturer={self.manufacturer!r}{extras}, "
                f"total={self._total}, allocated={self._allocated})"
            )

    def __str__(self) -> str:
        return self.name

## 3 · Subclasses: CPU, Storage, HDD and SSD

In [3]:
class CPU(Resource):
    """A CPU stock pool with immutable processor specifications."""

    __slots__ = ("_cores", "_socket", "_power_watts")

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        cores: int,
        socket: str,
        power_watts: int,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        cores = validate_integer(cores, name="cores", minimum=1)
        socket = validate_text(socket, name="socket")
        power_watts = validate_integer(power_watts, name="power_watts", minimum=1)
        super().__init__(name, manufacturer, total=total, allocated=allocated)
        self._cores = cores
        self._socket = socket
        self._power_watts = power_watts

    @property
    def cores(self) -> int:
        return self._cores

    @property
    def socket(self) -> str:
        return self._socket

    @property
    def power_watts(self) -> int:
        return self._power_watts

    def _spec_repr(self) -> str:
        return (
            f"cores={self.cores!r}, socket={self.socket!r}, "
            f"power_watts={self.power_watts!r}"
        )


class Storage(Resource):
    """Intermediate class shared by HDD and SSD."""

    __slots__ = ("_capacity_GB",)

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        capacity_GB: int,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        capacity_GB = validate_integer(capacity_GB, name="capacity_GB", minimum=1)
        super().__init__(name, manufacturer, total=total, allocated=allocated)
        self._capacity_GB = capacity_GB

    @property
    def capacity_GB(self) -> int:
        return self._capacity_GB

    def _spec_repr(self) -> str:
        return f"capacity_GB={self.capacity_GB!r}"


class HDD(Storage):
    """Mechanical disk inventory with form-factor and RPM specifications."""

    __slots__ = ("_size", "_rpm")

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        capacity_GB: int,
        size: str,
        rpm: int,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        size = validate_text(size, name="size")
        rpm = validate_integer(rpm, name="rpm", minimum=1)
        super().__init__(
            name, manufacturer, capacity_GB=capacity_GB,
            total=total, allocated=allocated,
        )
        self._size = size
        self._rpm = rpm

    @property
    def size(self) -> str:
        return self._size

    @property
    def rpm(self) -> int:
        return self._rpm

    def _spec_repr(self) -> str:
        return f"{super()._spec_repr()}, size={self.size!r}, rpm={self.rpm!r}"


class SSD(Storage):
    """Solid-state disk inventory with a read-only interface specification."""

    __slots__ = ("_interface",)

    def __init__(
        self,
        name: str,
        manufacturer: str,
        *,
        capacity_GB: int,
        interface: str,
        total: int = 0,
        allocated: int = 0,
    ) -> None:
        interface = validate_text(interface, name="interface")
        super().__init__(
            name, manufacturer, capacity_GB=capacity_GB,
            total=total, allocated=allocated,
        )
        self._interface = interface

    @property
    def interface(self) -> str:
        return self._interface

    def _spec_repr(self) -> str:
        return f"{super()._spec_repr()}, interface={self.interface!r}"

## 4 · Worked examples

In [4]:
cpu = CPU(
    "AMD Ryzen 7 2700", "AMD", cores=8, socket="AM4",
    power_watts=65, total=5, allocated=1,
)
ssd = SSD(
    "970 EVO", "Samsung", capacity_GB=1000,
    interface="PCIe NVMe 3.0 x4", total=4,
)
hdd = HDD(
    "BarraCuda", "Seagate", capacity_GB=2000,
    size='3.5"', rpm=7200, total=3,
)

assert str(cpu) == "AMD Ryzen 7 2700"
assert cpu.category == "cpu"
assert (cpu.total, cpu.allocated, cpu.available) == (5, 1, 4)
cpu.claim(2)
assert (cpu.total, cpu.allocated, cpu.available) == (5, 3, 2)
cpu.freeup(1)
cpu.died(1)  # one allocated CPU was damaged beyond repair
cpu.purchased(3)
assert (cpu.total, cpu.allocated, cpu.available) == (7, 1, 6)
assert isinstance(ssd, Storage) and isinstance(hdd, Storage)
assert eval(repr(cpu), {"CPU": CPU}) .snapshot() == cpu.snapshot()
print("CPU after inventory operations:", cpu.snapshot())
print("Storage examples:", repr(hdd), "|", repr(ssd))

CPU after inventory operations: InventorySnapshot(name='AMD Ryzen 7 2700', category='cpu', manufacturer='AMD', total=7, allocated=1, available=6)
Storage examples: HDD(name='BarraCuda', manufacturer='Seagate', capacity_GB=2000, size='3.5"', rpm=7200, total=3, allocated=0) | SSD(name='970 EVO', manufacturer='Samsung', capacity_GB=1000, interface='PCIe NVMe 3.0 x4', total=4, allocated=0)


## 5 · Automated tests

This suite uses `unittest` from the standard library, so the notebook runs
without installation. It covers validation, read-only APIs, inheritance,
atomic failures, representations and concurrent allocation. The suite runs
directly in the notebook, without relying on Jupyter's `unittest.main()` CLI.

In [5]:
import unittest
from concurrent.futures import ThreadPoolExecutor


class ValidatorTests(unittest.TestCase):
    def test_integer_limits_inclusive(self):
        self.assertEqual(validate_integer(2, name="x", minimum=2, maximum=4), 2)
        self.assertEqual(validate_integer(4, name="x", minimum=2, maximum=4), 4)

    def test_integer_rejects_float_string_bool(self):
        for value in (1.0, "1", True, False, None):
            with self.subTest(value=value), self.assertRaises(TypeError):
                validate_integer(value, name="x")

    def test_integer_minimum_and_custom_message(self):
        with self.assertRaisesRegex(ValueError, "custom"):
            validate_integer(0, name="x", minimum=1, bounds_message="custom")

    def test_integer_maximum(self):
        with self.assertRaises(ValueError):
            validate_integer(5, name="x", maximum=4)

    def test_text_normalizes_whitespace(self):
        self.assertEqual(validate_text("  AMD  ", name="x"), "AMD")

    def test_text_rejects_bad_inputs(self):
        for value, exception in ((5, TypeError), ("  ", ValueError)):
            with self.subTest(value=value), self.assertRaises(exception):
                validate_text(value, name="x")


class ResourceTests(unittest.TestCase):
    def setUp(self):
        self.resource = Resource("  Fan  ", "  Noctua  ", total=5, allocated=2)

    def test_initial_state_and_names(self):
        r = self.resource
        self.assertEqual((r.name, r.manufacturer), ("Fan", "Noctua"))
        self.assertEqual(r.manufacturer_name, "Noctua")
        self.assertEqual((r.total, r.allocated, r.available), (5, 2, 3))
        self.assertEqual(r.category, "resource")

    def test_default_counts(self):
        r = Resource("Fan", "Noctua")
        self.assertEqual((r.total, r.allocated, r.available), (0, 0, 0))

    def test_invalid_initial_counts(self):
        for kwargs in ({"total": -1}, {"allocated": -1}, {"total": 1, "allocated": 2}):
            with self.subTest(kwargs=kwargs), self.assertRaises(ValueError):
                Resource("Fan", "Noctua", **kwargs)

    def test_invalid_count_types(self):
        for kwargs in ({"total": True}, {"allocated": 1.5}):
            with self.subTest(kwargs=kwargs), self.assertRaises(TypeError):
                Resource("Fan", "Noctua", **kwargs)

    def test_reject_empty_names(self):
        with self.assertRaises(ValueError):
            Resource(" ", "Company")
        with self.assertRaises(ValueError):
            Resource("Item", " ")

    def test_properties_read_only(self):
        for attribute, value in (
            ("name", "Other"), ("manufacturer", "Other"),
            ("manufacturer_name", "Other"), ("total", 100),
            ("allocated", 100), ("available", 100), ("category", "other"),
        ):
            with self.subTest(attribute=attribute), self.assertRaises(AttributeError):
                setattr(self.resource, attribute, value)

    def test_claim_success(self):
        self.resource.claim(3)
        self.assertEqual((self.resource.allocated, self.resource.available), (5, 0))

    def test_claim_failure_is_atomic(self):
        before = self.resource.snapshot()
        with self.assertRaises(ValueError):
            self.resource.claim(4)
        self.assertEqual(self.resource.snapshot(), before)

    def test_freeup_success(self):
        self.resource.freeup(2)
        self.assertEqual((self.resource.allocated, self.resource.available), (0, 5))

    def test_freeup_failure_is_atomic(self):
        before = self.resource.snapshot()
        with self.assertRaises(ValueError):
            self.resource.freeup(3)
        self.assertEqual(self.resource.snapshot(), before)

    def test_died_retires_allocated(self):
        self.resource.died(2)
        self.assertEqual((self.resource.total, self.resource.allocated, self.resource.available), (3, 0, 3))

    def test_died_failure_is_atomic(self):
        before = self.resource.snapshot()
        with self.assertRaises(ValueError):
            self.resource.died(3)
        self.assertEqual(self.resource.snapshot(), before)

    def test_purchased(self):
        self.resource.purchased(2)
        self.assertEqual((self.resource.total, self.resource.allocated, self.resource.available), (7, 2, 5))

    def test_all_methods_reject_nonpositive_quantity(self):
        for method in ("claim", "freeup", "died", "purchased"):
            for quantity in (0, -1):
                with self.subTest(method=method, quantity=quantity), self.assertRaises(ValueError):
                    getattr(self.resource, method)(quantity)

    def test_all_methods_reject_bool_and_float(self):
        for method in ("claim", "freeup", "died", "purchased"):
            for quantity in (True, 1.2):
                with self.subTest(method=method, quantity=quantity), self.assertRaises(TypeError):
                    getattr(self.resource, method)(quantity)

    def test_snapshot_is_immutable(self):
        with self.assertRaises(AttributeError):
            self.resource.snapshot().total = 100

    def test_str_repr_round_trip(self):
        self.assertEqual(str(self.resource), "Fan")
        reconstructed = eval(repr(self.resource), {"Resource": Resource})
        self.assertEqual(reconstructed.snapshot(), self.resource.snapshot())

    def test_concurrent_claims_do_not_overallocate(self):
        resource = Resource("Fan", "Noctua", total=30)
        def try_claim(_):
            try:
                resource.claim(1)
                return True
            except ValueError:
                return False
        with ThreadPoolExecutor(max_workers=12) as pool:
            outcomes = list(pool.map(try_claim, range(100)))
        self.assertEqual(sum(outcomes), 30)
        self.assertEqual((resource.total, resource.allocated, resource.available), (30, 30, 0))


class SubclassTests(unittest.TestCase):
    def test_cpu_constructor_and_specs(self):
        cpu = CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65, total=5)
        self.assertIsInstance(cpu, Resource)
        self.assertEqual((cpu.cores, cpu.socket, cpu.power_watts), (8, "AM4", 65))
        self.assertEqual(cpu.category, "cpu")

    def test_cpu_specs_are_read_only(self):
        cpu = CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65)
        for key in ("cores", "socket", "power_watts"):
            with self.subTest(key=key), self.assertRaises(AttributeError):
                setattr(cpu, key, 99)

    def test_cpu_rejects_bad_specs(self):
        for overrides, error in (
            ({"cores": 0}, ValueError), ({"cores": True}, TypeError),
            ({"socket": "  "}, ValueError), ({"power_watts": -5}, ValueError),
        ):
            kwargs = {"cores": 8, "socket": "AM4", "power_watts": 65} | overrides
            with self.subTest(overrides=overrides), self.assertRaises(error):
                CPU("Processor", "AMD", **kwargs)

    def test_storage_inheritance_and_capacity(self):
        disk = Storage("Disk", "WD", capacity_GB=500)
        self.assertIsInstance(disk, Resource)
        self.assertEqual(disk.capacity_GB, 500)
        with self.assertRaises(AttributeError):
            disk.capacity_GB = 1000
        with self.assertRaises(ValueError):
            Storage("Disk", "WD", capacity_GB=0)

    def test_hdd_inherits_storage(self):
        disk = HDD("Disk", "WD", capacity_GB=2000, size='3.5"', rpm=7200)
        self.assertIsInstance(disk, Storage)
        self.assertEqual((disk.size, disk.rpm, disk.capacity_GB), ('3.5"', 7200, 2000))
        self.assertEqual(disk.category, "hdd")

    def test_hdd_bad_specs(self):
        with self.assertRaises(TypeError):
            HDD("Disk", "WD", capacity_GB=500, size='3.5"', rpm=True)
        with self.assertRaises(ValueError):
            HDD("Disk", "WD", capacity_GB=500, size=" ", rpm=7200)
        with self.assertRaises(ValueError):
            HDD("Disk", "WD", capacity_GB=500, size='3.5"', rpm=0)

    def test_ssd_inherits_storage(self):
        disk = SSD("Disk", "Samsung", capacity_GB=1000, interface="NVMe", total=2)
        self.assertIsInstance(disk, Storage)
        self.assertEqual((disk.interface, disk.capacity_GB, disk.category), ("NVMe", 1000, "ssd"))
        with self.assertRaises(AttributeError):
            disk.interface = "SATA"
        with self.assertRaises(ValueError):
            SSD("Disk", "Samsung", capacity_GB=1000, interface=" ")

    def test_subclass_repr_round_trip(self):
        items = [
            CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65, total=3, allocated=2),
            Storage("Disk", "WD", capacity_GB=500, total=2),
            HDD("Disk", "WD", capacity_GB=2000, size='3.5"', rpm=7200, total=2),
            SSD("Disk", "Samsung", capacity_GB=1000, interface="NVMe", total=2),
        ]
        for item in items:
            with self.subTest(cls=type(item).__name__):
                reconstructed = eval(repr(item), {
                    "CPU": CPU, "Storage": Storage, "HDD": HDD, "SSD": SSD,
                })
                self.assertEqual(reconstructed.snapshot(), item.snapshot())
                self.assertEqual(repr(reconstructed), repr(item))

    def test_subclass_inherits_inventory_methods(self):
        cpu = CPU("Processor", "AMD", cores=8, socket="AM4", power_watts=65, total=4)
        cpu.claim(2)
        cpu.died(1)
        cpu.freeup(1)
        cpu.purchased(2)
        self.assertEqual((cpu.total, cpu.allocated, cpu.available), (5, 0, 5))

In [6]:
suite = unittest.TestSuite(
    unittest.defaultTestLoader.loadTestsFromTestCase(cls)
    for cls in (ValidatorTests, ResourceTests, SubclassTests)
)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "One or more Project 3 tests failed"
print(f"PASS: {result.testsRun} automated tests")

test_integer_limits_inclusive (__main__.ValidatorTests.test_integer_limits_inclusive) ... ok
test_integer_maximum (__main__.ValidatorTests.test_integer_maximum) ... ok
test_integer_minimum_and_custom_message (__main__.ValidatorTests.test_integer_minimum_and_custom_message) ... ok
test_integer_rejects_float_string_bool (__main__.ValidatorTests.test_integer_rejects_float_string_bool) ... ok
test_text_normalizes_whitespace (__main__.ValidatorTests.test_text_normalizes_whitespace) ... ok
test_text_rejects_bad_inputs (__main__.ValidatorTests.test_text_rejects_bad_inputs) ... ok
test_all_methods_reject_bool_and_float (__main__.ResourceTests.test_all_methods_reject_bool_and_float) ... ok
test_all_methods_reject_nonpositive_quantity (__main__.ResourceTests.test_all_methods_reject_nonpositive_quantity) ... ok
test_claim_failure_is_atomic (__main__.ResourceTests.test_claim_failure_is_atomic) ... ok
test_claim_success (__main__.ResourceTests.test_claim_success) ... ok
test_concurrent_claims_do_no

PASS: 33 automated tests


## 6 · Engineering notes

- This is an in-memory class model: no database or durable state is promised.
- The `died` contract was ambiguous in the prompt. Here a damaged component
  is assumed to be **allocated**, and its removal decrements both counts.
  If unused stock must be disposed of, use the distinct `retire_available`
  operation in the **extended** notebook.
- `Storage` can be instantiated; making it abstract would add restrictions
  not requested by the exercise.
- Unit tests are executable specifications, and rerunning the notebook from
  a clean kernel should reproduce the same results.